# Task 1: Acquisition + Comparable Imagery + Initial Change Prototype

In [1]:
# pip install earthengine-api

In [2]:
import ee
import pandas as pd

In [3]:
ee.Authenticate()

True

In [4]:
ee.Initialize(project='multimodal-flood-risk')

print("Earth Engine initialized successfully.")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine initialized successfully.


## Define the 2 areas

In [5]:
# ------------------------------------------------------------
# PP1 Candidate AOI
# ------------------------------------------------------------

candidate_aoi = ee.Geometry.Rectangle([
    79.82,   # west
    6.80,    # south
    80.28,   # east
    7.25     # north
])


# ------------------------------------------------------------
# Small technical pilot AOI
# ------------------------------------------------------------

pilot_aoi = ee.Geometry.Rectangle([
    80.055,  # west
    6.885,  # south
    80.105, # east
    6.935   # north
])


print(
    "Candidate AOI area (km²):",
    candidate_aoi.area(1).divide(1_000_000).getInfo()
)

print(
    "Hanwella pilot area (km²):",
    pilot_aoi.area(1).divide(1_000_000).getInfo()
)

Candidate AOI area (km²): 2540.210940965659
Hanwella pilot area (km²): 30.686324537958345


## Defining the EV2 search window

In [6]:
SEARCH_START = '2025-11-20'
SEARCH_END = '2025-12-24'

print("Sentinel-1 search window:")
print(SEARCH_START, "to", SEARCH_END)

Sentinel-1 search window:
2025-11-20 to 2025-12-24


## Retrieving Sentinel-1

In [7]:
s1_collection = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(candidate_aoi)
    .filterDate(SEARCH_START, SEARCH_END)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(
        ee.Filter.listContains(
            'transmitterReceiverPolarisation',
            'VV'
        )
    )
    .filter(
        ee.Filter.listContains(
            'transmitterReceiverPolarisation',
            'VH'
        )
    )
    .sort('system:time_start')
)

s1_count = s1_collection.size().getInfo()

print("Number of Sentinel-1 candidate scenes:", s1_count)

Number of Sentinel-1 candidate scenes: 14


## Sentinel‑1 Metadata Inventory

In [8]:
def extract_s1_metadata(image):
    """
    Extract the metadata needed for PP1 Sentinel-1
    acquisition screening.
    """

    acquisition_time = ee.Date(
        image.get('system:time_start')
    )

    intersection_area = (
        image.geometry()
        .intersection(candidate_aoi, 1)
        .area(1)
    )

    candidate_area = candidate_aoi.area(1)

    coverage_percent = (
        intersection_area
        .divide(candidate_area)
        .multiply(100)
    )

    return ee.Feature(None, {

        'scene_id':
            image.get('system:index'),

        'asset_id':
            image.id(),

        'acquisition_utc':
            acquisition_time.format(
                'yyyy-MM-dd HH:mm',
                'UTC'
            ),

        'acquisition_local':
            acquisition_time.format(
                'yyyy-MM-dd HH:mm',
                'Asia/Colombo'
            ),

        'platform':
            image.get('platform_number'),

        'orbit_pass':
            image.get(
                'orbitProperties_pass'
            ),

        'relative_orbit':
            image.get(
                'relativeOrbitNumber_start'
            ),

        'instrument_mode':
            image.get(
                'instrumentMode'
            ),

        'polarization':
            ee.List(
                image.get(
                    'transmitterReceiverPolarisation'
                )
            ).join(','),

        'resolution_m':
            image.get(
                'resolution_meters'
            ),

        'slice_number':
            image.get(
                'sliceNumber'
            ),

        'total_slices':
            image.get(
                'totalSlices'
            ),

        'coverage_percent':
            coverage_percent

    })

In [9]:
inventory_fc = ee.FeatureCollection(
    s1_collection.map(extract_s1_metadata)
)

inventory_data = inventory_fc.getInfo()

rows = [
    feature['properties']
    for feature in inventory_data['features']
]

inventory_df = pd.DataFrame(rows)

inventory_df

,acquisition_local,acquisition_utc,asset_id,coverage_percent,instrument_mode,orbit_pass,platform,polarization,relative_orbit,resolution_m,scene_id,slice_number,total_slices
0,2025-11-26 05:55,2025-11-26 00:25,S1A_IW_GRDH_1SDV_20251126T002515_20251126T0025...,56.693060,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251126T002515_20251126T0025...,2,3
1,2025-11-26 05:55,2025-11-26 00:25,S1A_IW_GRDH_1SDV_20251126T002540_20251126T0026...,43.373912,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251126T002540_20251126T0026...,3,3
2,2025-11-26 18:19,2025-11-26 12:49,S1A_IW_GRDH_1SDV_20251126T124916_20251126T1249...,6.576538,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251126T124916_20251126T1249...,1,3
3,2025-11-26 18:19,2025-11-26 12:49,S1A_IW_GRDH_1SDV_20251126T124945_20251126T1250...,3.084946,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251126T124945_20251126T1250...,2,3
4,2025-12-02 05:54,2025-12-02 00:24,S1C_IW_GRDH_1SDV_20251202T002429_20251202T0024...,100.000000,IW,DESCENDING,C,"VV,VH",19,10,S1C_IW_GRDH_1SDV_20251202T002429_20251202T0024...,3,4
5,2025-12-02 18:18,2025-12-02 12:48,S1C_IW_GRDH_1SDV_20251202T124832_20251202T1248...,9.728797,IW,ASCENDING,C,"VV,VH",27,10,S1C_IW_GRDH_1SDV_20251202T124832_20251202T1248...,2,6
6,2025-12-08 05:55,2025-12-08 00:25,S1A_IW_GRDH_1SDV_20251208T002514_20251208T0025...,56.733736,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251208T002514_20251208T0025...,2,3
7,2025-12-08 05:55,2025-12-08 00:25,S1A_IW_GRDH_1SDV_20251208T002539_20251208T0026...,43.333318,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251208T002539_20251208T0026...,3,3
8,2025-12-08 18:19,2025-12-08 12:49,S1A_IW_GRDH_1SDV_20251208T124915_20251208T1249...,6.686733,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251208T124915_20251208T1249...,1,3
9,2025-12-08 18:19,2025-12-08 12:49,S1A_IW_GRDH_1SDV_20251208T124944_20251208T1250...,3.108311,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251208T124944_20251208T1250...,2,3


In [10]:
inventory_df = inventory_df.sort_values(
    'acquisition_local'
).reset_index(drop=True)

inventory_df

,acquisition_local,acquisition_utc,asset_id,coverage_percent,instrument_mode,orbit_pass,platform,polarization,relative_orbit,resolution_m,scene_id,slice_number,total_slices
0,2025-11-26 05:55,2025-11-26 00:25,S1A_IW_GRDH_1SDV_20251126T002515_20251126T0025...,56.693060,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251126T002515_20251126T0025...,2,3
1,2025-11-26 05:55,2025-11-26 00:25,S1A_IW_GRDH_1SDV_20251126T002540_20251126T0026...,43.373912,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251126T002540_20251126T0026...,3,3
2,2025-11-26 18:19,2025-11-26 12:49,S1A_IW_GRDH_1SDV_20251126T124916_20251126T1249...,6.576538,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251126T124916_20251126T1249...,1,3
3,2025-11-26 18:19,2025-11-26 12:49,S1A_IW_GRDH_1SDV_20251126T124945_20251126T1250...,3.084946,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251126T124945_20251126T1250...,2,3
4,2025-12-02 05:54,2025-12-02 00:24,S1C_IW_GRDH_1SDV_20251202T002429_20251202T0024...,100.000000,IW,DESCENDING,C,"VV,VH",19,10,S1C_IW_GRDH_1SDV_20251202T002429_20251202T0024...,3,4
5,2025-12-02 18:18,2025-12-02 12:48,S1C_IW_GRDH_1SDV_20251202T124832_20251202T1248...,9.728797,IW,ASCENDING,C,"VV,VH",27,10,S1C_IW_GRDH_1SDV_20251202T124832_20251202T1248...,2,6
6,2025-12-08 05:55,2025-12-08 00:25,S1A_IW_GRDH_1SDV_20251208T002514_20251208T0025...,56.733736,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251208T002514_20251208T0025...,2,3
7,2025-12-08 05:55,2025-12-08 00:25,S1A_IW_GRDH_1SDV_20251208T002539_20251208T0026...,43.333318,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251208T002539_20251208T0026...,3,3
8,2025-12-08 18:19,2025-12-08 12:49,S1A_IW_GRDH_1SDV_20251208T124915_20251208T1249...,6.686733,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251208T124915_20251208T1249...,1,3
9,2025-12-08 18:19,2025-12-08 12:49,S1A_IW_GRDH_1SDV_20251208T124944_20251208T1250...,3.108311,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251208T124944_20251208T1250...,2,3


In [11]:
inventory_df[
    [
        'scene_id',
        'acquisition_local',
        'platform',
        'orbit_pass',
        'relative_orbit',
        'instrument_mode',
        'polarization',
        'coverage_percent'
    ]
]

,scene_id,acquisition_local,platform,orbit_pass,relative_orbit,instrument_mode,polarization,coverage_percent
0,S1A_IW_GRDH_1SDV_20251126T002515_20251126T0025...,2025-11-26 05:55,A,DESCENDING,19,IW,"VV,VH",56.693060
1,S1A_IW_GRDH_1SDV_20251126T002540_20251126T0026...,2025-11-26 05:55,A,DESCENDING,19,IW,"VV,VH",43.373912
2,S1A_IW_GRDH_1SDV_20251126T124916_20251126T1249...,2025-11-26 18:19,A,ASCENDING,27,IW,"VV,VH",6.576538
3,S1A_IW_GRDH_1SDV_20251126T124945_20251126T1250...,2025-11-26 18:19,A,ASCENDING,27,IW,"VV,VH",3.084946
4,S1C_IW_GRDH_1SDV_20251202T002429_20251202T0024...,2025-12-02 05:54,C,DESCENDING,19,IW,"VV,VH",100.000000
5,S1C_IW_GRDH_1SDV_20251202T124832_20251202T1248...,2025-12-02 18:18,C,ASCENDING,27,IW,"VV,VH",9.728797
6,S1A_IW_GRDH_1SDV_20251208T002514_20251208T0025...,2025-12-08 05:55,A,DESCENDING,19,IW,"VV,VH",56.733736
7,S1A_IW_GRDH_1SDV_20251208T002539_20251208T0026...,2025-12-08 05:55,A,DESCENDING,19,IW,"VV,VH",43.333318
8,S1A_IW_GRDH_1SDV_20251208T124915_20251208T1249...,2025-12-08 18:19,A,ASCENDING,27,IW,"VV,VH",6.686733
9,S1A_IW_GRDH_1SDV_20251208T124944_20251208T1250...,2025-12-08 18:19,A,ASCENDING,27,IW,"VV,VH",3.108311


In [12]:
inventory_df.insert(
    0,
    'event_id',
    'EV2_Nov_2025'
)

inventory_df.insert(
    1,
    'aoi_id',
    'PP1_AOI_KELANI_01'
)

inventory_df

,event_id,aoi_id,acquisition_local,acquisition_utc,asset_id,coverage_percent,instrument_mode,orbit_pass,platform,polarization,relative_orbit,resolution_m,scene_id,slice_number,total_slices
0,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-11-26 05:55,2025-11-26 00:25,S1A_IW_GRDH_1SDV_20251126T002515_20251126T0025...,56.693060,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251126T002515_20251126T0025...,2,3
1,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-11-26 05:55,2025-11-26 00:25,S1A_IW_GRDH_1SDV_20251126T002540_20251126T0026...,43.373912,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251126T002540_20251126T0026...,3,3
2,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-11-26 18:19,2025-11-26 12:49,S1A_IW_GRDH_1SDV_20251126T124916_20251126T1249...,6.576538,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251126T124916_20251126T1249...,1,3
3,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-11-26 18:19,2025-11-26 12:49,S1A_IW_GRDH_1SDV_20251126T124945_20251126T1250...,3.084946,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251126T124945_20251126T1250...,2,3
4,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-12-02 05:54,2025-12-02 00:24,S1C_IW_GRDH_1SDV_20251202T002429_20251202T0024...,100.000000,IW,DESCENDING,C,"VV,VH",19,10,S1C_IW_GRDH_1SDV_20251202T002429_20251202T0024...,3,4
5,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-12-02 18:18,2025-12-02 12:48,S1C_IW_GRDH_1SDV_20251202T124832_20251202T1248...,9.728797,IW,ASCENDING,C,"VV,VH",27,10,S1C_IW_GRDH_1SDV_20251202T124832_20251202T1248...,2,6
6,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-12-08 05:55,2025-12-08 00:25,S1A_IW_GRDH_1SDV_20251208T002514_20251208T0025...,56.733736,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251208T002514_20251208T0025...,2,3
7,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-12-08 05:55,2025-12-08 00:25,S1A_IW_GRDH_1SDV_20251208T002539_20251208T0026...,43.333318,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251208T002539_20251208T0026...,3,3
8,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-12-08 18:19,2025-12-08 12:49,S1A_IW_GRDH_1SDV_20251208T124915_20251208T1249...,6.686733,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251208T124915_20251208T1249...,1,3
9,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-12-08 18:19,2025-12-08 12:49,S1A_IW_GRDH_1SDV_20251208T124944_20251208T1250...,3.108311,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251208T124944_20251208T1250...,2,3


## Saving the inventory

In [13]:
inventory_path = (
    "../shared/events/s1_scene_inventory_ev2_pp1_bt_mask_pilot.csv"
)

inventory_df.to_csv(
    inventory_path,
    index=False
)

print(
    f"Saved Sentinel-1 inventory to: {inventory_path}"
)

Saved Sentinel-1 inventory to: ../shared/events/s1_scene_inventory_ev2_pp1_bt_mask_pilot.csv


## Deciding the exact candidate pair

In [14]:
PRE_SCENE_IDS = [
    "S1A_IW_GRDH_1SDV_20251126T002515_20251126T002540_062041_07C2FF_C9CA",
    "S1A_IW_GRDH_1SDV_20251126T002540_20251126T002605_062041_07C2FF_436D",
]

POST_SCENE_ID = (
    "S1C_IW_GRDH_1SDV_20251202T002429_20251202T002454_005265_00A742_E36B"
)

print("Pre-event slices:")
for scene_id in PRE_SCENE_IDS:
    print(" ", scene_id)

print("\nPost-event scene:")
print(" ", POST_SCENE_ID)

Pre-event slices:
  S1A_IW_GRDH_1SDV_20251126T002515_20251126T002540_062041_07C2FF_C9CA
  S1A_IW_GRDH_1SDV_20251126T002540_20251126T002605_062041_07C2FF_436D

Post-event scene:
  S1C_IW_GRDH_1SDV_20251202T002429_20251202T002454_005265_00A742_E36B


## Building the pre-event image from its two slices

In [15]:
pre_images = [
    ee.Image(f"COPERNICUS/S1_GRD/{scene_id}")
    for scene_id in PRE_SCENE_IDS
]

pre_collection = ee.ImageCollection.fromImages(pre_images)

pre_image = pre_collection.mosaic().clip(pilot_aoi)

print("Pre-event image prepared.")

Pre-event image prepared.


In [16]:
# AOI area
aoi_area = pilot_aoi.area()

# Intersection area between mosaic and AOI
mosaic_area = pre_image.geometry().intersection(pilot_aoi).area()

# Coverage percent
coverage_percent = mosaic_area.divide(aoi_area).multiply(100)

print("Coverage percent:", coverage_percent.getInfo())

Coverage percent: 100


## Loading the post-event image

In [17]:
post_image = (
    ee.Image(f"COPERNICUS/S1_GRD/{POST_SCENE_ID}")
    .clip(pilot_aoi)
)

print("Post-event image prepared.")

Post-event image prepared.


In [18]:
# AOI area
aoi_area = pilot_aoi.area()

# Intersection area between mosaic and AOI
mosaic_area = post_image.geometry().intersection(pilot_aoi).area()

# Coverage percent
coverage_percent = mosaic_area.divide(aoi_area).multiply(100)

print("Coverage percent:", coverage_percent.getInfo())

Coverage percent: 100


## Using the same ploarisation

In [19]:
pre_vv = pre_image.select("VV")
post_vv = post_image.select("VV")

pre_vh = pre_image.select("VH")
post_vh = post_image.select("VH")

print("VV and VH bands selected consistently for both dates.")

VV and VH bands selected consistently for both dates.


## Creating the change image

In [20]:
# change_vv_db = post_vv.subtract(pre_vv).rename("VV_change_dB")

# stats = change_vv_db.reduceRegion(
#     reducer=ee.Reducer.percentile([2, 50, 98]),
#     geometry=pilot_aoi,
#     scale=10,
#     bestEffort=True,
#     maxPixels=1e8
# )

# print(stats.getInfo())

## Displaying pre, post and change consistently

In [21]:
# !pip install geemap


In [22]:
import geemap

In [23]:
# Map = geemap.Map()

# Map.centerObject(pilot_aoi, 13)

# vv_vis = {
#     "min": -25,
#     "max": 5,
#     "bands": ["VV"]
# }

# change_vis = {
#     "min": -5,
#     "max": 5,
#     "bands": ["VV_change_dB"],
#     "palette": ["red", "white", "blue"]
# }

# Map.addLayer(pre_vv, vv_vis, "Pre-event VV — 26 Nov 2025")
# Map.addLayer(post_vv, vv_vis, "Post-event VV — 02 Dec 2025")
# Map.addLayer(change_vv_db, change_vis, "VV change — Post minus Pre")

# Map.addLayer(
#     pilot_aoi,
#     {"color": "yellow"},
#     "Pilot AOI"
# )

# Map

## Making a provisional candidate change mask

In [24]:
# candidate_mask = change_vv_db.lt(-3).rename("candidate_flood_change")

In [25]:
# Map.addLayer(
#     candidate_mask.selfMask(),
#     {"palette": ["cyan"]},
#     "Change mask (-3 dB)"
# )

# Task 2: Thresholding, Morphology, Permanent‑Water Masking, QA

## Log‑ratio calculation

In [26]:
# NEW canonical log-ratio definition
# COPERNICUS/S1_GRD VV values are in dB.
# Positive values mean post-event backscatter is lower than pre-event.
log_ratio_db = (
    pre_vv.subtract(post_vv)
    .rename("VV_log_ratio_dB")
)

stats = log_ratio_db.reduceRegion(
    reducer=ee.Reducer.percentile([2, 50, 98]),
    geometry=pilot_aoi,
    scale=10,
    bestEffort=True,
    maxPixels=1e8
)
print(stats.getInfo())

# Map visualization with log-ratio
Map = geemap.Map()
Map.centerObject(pilot_aoi, 13)

vv_vis = {"min": -25, "max": 5, "bands": ["VV"]}
logratio_vis = {"min": -5, "max": 5, "bands": ["VV_log_ratio_dB"], "palette": ["red","white","blue"]}

Map.addLayer(pre_vv, vv_vis, "Pre-event VV — 26 Nov 2025")
Map.addLayer(post_vv, vv_vis, "Post-event VV — 02 Dec 2025")
Map.addLayer(log_ratio_db, logratio_vis, "Pre/Post log-ratio — dB")
Map.addLayer(pilot_aoi, {"color": "yellow"}, "Pilot AOI")

{'VV_log_ratio_dB_p2': -5.622215409956864, 'VV_log_ratio_dB_p50': 0.37467052622419417, 'VV_log_ratio_dB_p98': 10.871047029229763}


positive → decrease after event
negative → increase after event

## Pilot-only fixed 3 dB threshold test

In [27]:
candidate_mask_fixed = (
    log_ratio_db.gt(3)
    .rename("fixed_decrease_gt_3dB")
)

In [28]:
Map.addLayer(
    candidate_mask_fixed.selfMask(),
    {"palette": ["cyan"]},  
    "Fixed mask (>3 dB)"
)

## Otsu thresholding

In [29]:
# hist_dict = log_ratio_db.reduceRegion(
#     reducer=ee.Reducer.histogram(
#         maxBuckets=256,
#         minBucketWidth=0.1
#     ),
#     geometry=pilot_aoi,
#     scale=10,
#     bestEffort=True,
#     maxPixels=1e8
# ).getInfo()

# hist = hist_dict["VV_log_ratio_dB"]

# counts = hist["histogram"]
# bucket_means = hist["bucketMeans"]

In [30]:
# import numpy as np

# counts = np.asarray(counts, dtype=float)
# bucket_means = np.asarray(bucket_means, dtype=float)

# prob = counts / counts.sum()

# omega = np.cumsum(prob)
# mu = np.cumsum(prob * bucket_means)
# mu_total = mu[-1]

# between_class_variance = (
#     (mu_total * omega - mu) ** 2
#     / (omega * (1 - omega) + 1e-12)
# )

# otsu_index = np.nanargmax(between_class_variance)
# otsu_threshold = bucket_means[otsu_index]

# print("Otsu threshold (dB):", otsu_threshold)

In [31]:
# candidate_mask_otsu = log_ratio_db.gt(otsu_threshold)

In [32]:
# # Add Otsu candidate mask (before cleanup and permanent-water exclusion)
# Map.addLayer(
#     candidate_mask_otsu.selfMask(),
#     {"palette": ["orange"]},  
#     "Otsu mask (raw)"
# )

# Map.centerObject(pilot_aoi, 13) 

In [33]:
import numpy as np

hist_dict = log_ratio_db.reduceRegion(
    reducer=ee.Reducer.histogram(maxBuckets=256, minBucketWidth=0.1),
    geometry=pilot_aoi,
    scale=10,
    maxPixels=1e8,
    bestEffort=True
).getInfo()

hist = hist_dict.get("VV_log_ratio_dB")
counts = np.asarray(hist["histogram"], dtype=float)
bucket_means = np.asarray(hist["bucketMeans"], dtype=float)

prob = counts / counts.sum()
omega = np.cumsum(prob)
mu = np.cumsum(prob * bucket_means)
mu_total = mu[-1]

valid_splits = (omega > 0) & (omega < 1)
between_class_variance = np.full_like(omega, np.nan, dtype=float)
between_class_variance[valid_splits] = (
    (mu_total * omega[valid_splits] - mu[valid_splits]) ** 2
    / (omega[valid_splits] * (1 - omega[valid_splits]))
)

otsu_index = int(np.nanargmax(between_class_variance))
otsu_threshold = float(bucket_means[otsu_index])
print("Otsu threshold (dB):", otsu_threshold)

candidate_mask_otsu = (
    log_ratio_db.gt(otsu_threshold)
    .rename("otsu_candidate")
)


Otsu threshold (dB): 1.4998743861586086


In [34]:
Map.addLayer(
    candidate_mask_otsu.selfMask(),
    {"palette": ["orange"]},   
    "Otsu mask (raw)"
)

## Pixel‑selection summary (diagnostic)

In [35]:
valid_change = log_ratio_db.mask().reduce(ee.Reducer.min()).gt(0)

def selected_percent(binary_mask, output_name):
    binary_valid = (
        binary_mask.unmask(0)
        .updateMask(valid_change)
        .rename("selected")
    )
    result = binary_valid.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=pilot_aoi,
        scale=10,
        maxPixels=1e8,
        bestEffort=True
    ).get("selected")
    fraction = ee.Number(result).getInfo()
    print(f"{output_name}: {fraction * 100:.2f}% of valid change pixels selected")

# Run diagnostics
selected_percent(candidate_mask_fixed, "Fixed threshold > 3 dB")
selected_percent(candidate_mask_otsu, "Raw Otsu")

Fixed threshold > 3 dB: 19.49% of valid change pixels selected
Raw Otsu: 34.07% of valid change pixels selected


## Morphological cleanup

In [36]:
opened_mask = (
    candidate_mask_otsu
    .focal_min(radius=1, units="pixels")
    .focal_max(radius=1, units="pixels")
    .rename("otsu_after_opening")
)

clean_mask = (
    opened_mask
    .focal_max(radius=1, units="pixels")
    .focal_min(radius=1, units="pixels")
    .rename("otsu_after_opening_closing")
)

# Diagnostics
selected_percent(opened_mask, "After opening")
selected_percent(clean_mask, "After opening + closing")

After opening: 22.59% of valid change pixels selected
After opening + closing: 23.67% of valid change pixels selected


In [37]:
Map.addLayer(
    clean_mask.selfMask(),
    {"palette": ["blue"]},   
    "Otsu mask (clean)"
)

Map.centerObject(pilot_aoi, 13) 
Map

Map(center=[6.9100002107571905, 80.08000000000519], controls=(WidgetControl(options=['position', 'transparent_…

## Permanent‑water masking

In [38]:

# ============================================================
# PP1 EV2: JRC HISTORICAL PERSISTENT-WATER EXCLUSION
# ============================================================

from IPython.display import display

jrc_gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")

# JRC bands
jrc_occurrence = jrc_gsw.select("occurrence")
jrc_transition = jrc_gsw.select("transition")
jrc_seasonality = jrc_gsw.select("seasonality")

jrc_projection = jrc_transition.projection()

# Persistent water = transition classes 1 (Permanent), 2 (New permanent), 7 (Seasonal→Permanent)
persistent_transition_test = (
    jrc_transition.eq(1)
    .Or(jrc_transition.eq(2))
    .Or(jrc_transition.eq(7))
)

# Proxy mask: 1 = persistent water, 0 = other
persistent_water_proxy = (
    ee.Image.constant(0)
    .toByte()
    .setDefaultProjection(jrc_projection)
    .where(persistent_transition_test, 1)
    .clip(pilot_aoi)
    .rename("historical_persistent_water")
)

# Diagnostic: seasonality = 12 months
seasonality_12 = (
    ee.Image.constant(0)
    .toByte()
    .setDefaultProjection(jrc_seasonality.projection())
    .where(jrc_seasonality.eq(12), 1)
    .clip(pilot_aoi)
    .rename("water_seasonality_12")
)

# Diagnostic: occurrence ≥90%
occurrence_ge90 = (
    ee.Image.constant(0)
    .toByte()
    .setDefaultProjection(jrc_occurrence.projection())
    .where(jrc_occurrence.gte(90), 1)
    .clip(pilot_aoi)
    .rename("occurrence_ge90")
)

# ------------------------------------------------------------
# 2. Count selected pixels on the JRC native grid (~30 m)
# ------------------------------------------------------------

def count_selected_pixels(binary_image, band_name):
    result = binary_image.selfMask().reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=pilot_aoi,
        scale=30,
        crs=jrc_projection,
        maxPixels=1e8,
        tileScale=4
    ).getInfo()

    value = result.get(band_name)
    return int(value) if value is not None else 0

# Transition classes
class1_count = count_selected_pixels(
    jrc_transition.eq(1).rename("class1")
        .rename("class1"),
    "class1"
)
class2_count = count_selected_pixels(
    jrc_transition.eq(2).rename("class2"),
    "class2"
)
class7_count = count_selected_pixels(
    jrc_transition.eq(7).rename("class7"),
    "class7"
)

# Combined proxy and diagnostics
persistent_count = count_selected_pixels(
    persistent_water_proxy,
    "historical_persistent_water"
)
seasonality_count = count_selected_pixels(
    seasonality_12,
    "water_seasonality_12"
)
occurrence90_count = count_selected_pixels(
    occurrence_ge90,
    "occurrence_ge90"
)

print("JRC transition class 1 — Permanent:", class1_count)
print("JRC transition class 2 — New permanent:", class2_count)
print("JRC transition class 7 — Seasonal to permanent:", class7_count)
print("Combined persistent-water proxy:", persistent_count)
print("Seasonality = 12 months (comparison only):", seasonality_count)
print("Occurrence >= 90% (diagnostic only):", occurrence90_count)

# ------------------------------------------------------------
# 3. Apply the exclusion to the existing cleaned Otsu mask
# ------------------------------------------------------------

non_persistent_water = (
    persistent_water_proxy.Not()
    .rename("not_historical_persistent_water")
)

flood_mask_final_provisional = (
    clean_mask
    .updateMask(non_persistent_water)
    .rename("provisional_flood_mask")
)

# ---------------------------------------------------------------------------------
# 4. Quantify effect on the cleaned Otsu candidates count on Sentinel-1 grid (10 m)
# ---------------------------------------------------------------------------------

def count_s1_selected_pixels(binary_image, band_name):
    result = binary_image.selfMask().reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=pilot_aoi,
        scale=10,
        crs=log_ratio_db.projection(),
        maxPixels=1e8,
        tileScale=4
    ).getInfo()

    value = result.get(band_name)
    return int(value) if value is not None else 0

# Counts
candidate_pixels = count_s1_selected_pixels(
    clean_mask.rename("candidate"),
    "candidate"
)
excluded_candidate_overlap = count_s1_selected_pixels(
    clean_mask.And(persistent_water_proxy)
        .rename("excluded_overlap"),
    "excluded_overlap"
)
remaining_pixels = count_s1_selected_pixels(
    flood_mask_final_provisional.rename("remaining"),
    "remaining"
)

excluded_percentage = (
    100 * excluded_candidate_overlap / candidate_pixels
    if candidate_pixels else 0
)

print("\n--- Effect on cleaned Otsu candidates ---")
print("Cleaned candidate pixels:", candidate_pixels)
print("Candidate pixels overlapping historical water:",
      excluded_candidate_overlap)
print(f"Candidate pixels excluded: {excluded_percentage:.2f}%")
print("Remaining candidate pixels:", remaining_pixels)

# ------------------------------------------------------------
# 5. Map A: water-data diagnostics (only 3 layers)
# ------------------------------------------------------------

Map_jrc_diagnostic = geemap.Map()
Map_jrc_diagnostic.centerObject(pilot_aoi, 13)

Map_jrc_diagnostic.addLayer(
    jrc_occurrence,
    {
        "min": 0,
        "max": 100,
        "palette": ["white", "lightblue", "blue"]
    },
    "JRC occurrence (%)"
)

Map_jrc_diagnostic.addLayer(
    persistent_water_proxy.selfMask(),
    {"palette": ["magenta"]},
    "Persistent-water proxy (classes 1, 2, 7)"
)

Map_jrc_diagnostic.addLayer(
    seasonality_12.selfMask(),
    {"palette": ["cyan"]},
    "Seasonality = 12 months (comparison)"
)

display(Map_jrc_diagnostic)

# ------------------------------------------------------------
# 6. Map B: effect on the candidate mask (few layers)
# ------------------------------------------------------------

Map_mask_result = geemap.Map()
Map_mask_result.centerObject(pilot_aoi, 13)

Map_mask_result.addLayer(
    clean_mask.selfMask(),
    {"palette": ["blue"]},
    "Clean Otsu candidates (before exclusion)"
)

Map_mask_result.addLayer(
    flood_mask_final_provisional.selfMask(),
    {"palette": ["red"]},
    "Remaining candidates after exclusion"
)

Map_mask_result.addLayer(
    persistent_water_proxy.selfMask(),
    {"palette": ["magenta"]},
    "Historical water proxy used for exclusion"
)

Map_mask_result.addLayer(
    pilot_aoi,
    {"color": "yellow"},
    "Pilot AOI"
)

display(Map_mask_result)

JRC transition class 1 — Permanent: 148
JRC transition class 2 — New permanent: 25
JRC transition class 7 — Seasonal to permanent: 23
Combined persistent-water proxy: 196
Seasonality = 12 months (comparison only): 196
Occurrence >= 90% (diagnostic only): 0

--- Effect on cleaned Otsu candidates ---
Cleaned candidate pixels: 72263
Candidate pixels overlapping historical water: 570
Candidate pixels excluded: 0.79%
Remaining candidate pixels: 71693


Map(center=[6.9100002107571905, 80.08000000000519], controls=(WidgetControl(options=['position', 'transparent_…

Map(center=[6.9100002107571905, 80.08000000000519], controls=(WidgetControl(options=['position', 'transparent_…